# Skin Lesion Classification — 5-Class FAST Version

A lighter, quicker version of the full 9-class pipeline: same three tables,
but restricted to **5 classes** and **fewer epochs** so a full run finishes
much faster (good for a first pass / sanity check before the full run).

1. **Table 1** — Transfer learning models (AlexNet → EfficientNet-B0)
2. **Table 2** — Classical classifiers on deep features
3. **Table 3** — Computational efficiency comparison

Dataset: https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic
(download it yourself, unzip, upload to Drive — no Kaggle API needed here).

Run cells top to bottom. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

## 1. Install dependencies

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic


In [2]:
!pip install -q thop xgboost


## 2. Dataset — mount Drive and set the path

Download the dataset yourself from Kaggle:
https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic

Unzip it and upload the folder to your Google Drive (or upload the zip and
unzip it directly in Colab — see the commented option below). Then set
`DATA_ROOT` in the next cell to wherever the `Train`/`Test` subfolders end up.

In [4]:

DATA_ROOT = "/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration"

# --- Option B: you have the zip file in Drive and want to unzip it into Colab's
# local disk (faster I/O than reading straight off Drive during training) ---
# import zipfile
# zip_path = "/content/drive/MyDrive/skin-cancer9-classesisic.zip"
# with zipfile.ZipFile(zip_path, "r") as zf:
#     zf.extractall("./data")
# DATA_ROOT = "./data/Skin cancer ISIC The International Skin Imaging Collaboration"

import os
print("Contents of DATA_ROOT:", os.listdir(DATA_ROOT))


Contents of DATA_ROOT: ['Test', 'Train']


## 3. Config — edit hyperparameters here if needed

In [5]:
import os

TRAIN_DIR = os.path.join(DATA_ROOT, "Train")
TEST_DIR = os.path.join(DATA_ROOT, "Test")
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Train classes found:", os.listdir(TRAIN_DIR))
print("Test classes found:", os.listdir(TEST_DIR))

# --- 5-class subset (edit this list to pick different classes) ---
# Chosen for a clinically meaningful mix of malignant + benign + decent sample counts:
CLASS_SUBSET = [
    "nevus",
    "melanoma",
    "basal cell carcinoma",
    "pigmented benign keratosis",
    "squamous cell carcinoma",
]

VAL_FRACTION_OF_TRAIN = 0.15
RANDOM_SEED = 42

IMAGE_SIZE = 224
BATCH_SIZE = 32
FEATURE_EXTRACT_EPOCHS = 2     # reduced from 5 for speed
FINE_TUNE_EPOCHS = 6           # reduced from 20 for speed
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

CNN_MODELS = [
    "alexnet", "vgg16", "vgg19", "resnet18",
    "resnet50", "resnet101", "densenet121", "efficientnet_b0",
]

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("Using class subset:", CLASS_SUBSET)


Train classes found: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']
Test classes found: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']
Using device: cuda
Using class subset: ['nevus', 'melanoma', 'basal cell carcinoma', 'pigmented benign keratosis', 'squamous cell carcinoma']


## 4. Dataset loading — filtered to the 5-class subset

Unlike the full pipeline (which uses every subfolder via `ImageFolder`), this
version only scans the 5 folders listed in `CLASS_SUBSET` above.

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision.datasets.folder import default_loader
import torchvision.transforms as T

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")


def get_transforms(train=True):
    normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if train:
        return T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.2),
            T.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
            T.ToTensor(),
            normalize,
        ])
    return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(), normalize])


def scan_filtered_samples(root, allowed_classes):
    """Like torchvision.datasets.ImageFolder, but only scans the given
    subfolders instead of every subfolder under root."""
    classes = sorted(allowed_classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    samples = []
    for c in classes:
        class_dir = os.path.join(root, c)
        if not os.path.isdir(class_dir):
            raise FileNotFoundError(f"Expected class folder not found: {class_dir}")
        for fname in sorted(os.listdir(class_dir)):
            if fname.lower().endswith(IMG_EXTENSIONS):
                samples.append((os.path.join(class_dir, fname), class_to_idx[c]))
    return samples, classes, class_to_idx


class FilteredImageDataset(Dataset):
    """Dataset over a pre-scanned (path, label) samples list, restricted to
    CLASS_SUBSET, with a configurable transform."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = default_loader(path)
        if self.transform:
            image = self.transform(image)
        return image, label


def load_datasets():
    train_samples_all, classes, class_to_idx = scan_filtered_samples(TRAIN_DIR, CLASS_SUBSET)
    test_samples, _, _ = scan_filtered_samples(TEST_DIR, CLASS_SUBSET)

    labels = np.array([lbl for _, lbl in train_samples_all])
    indices = np.arange(len(train_samples_all))
    train_idx, val_idx = train_test_split(
        indices, test_size=VAL_FRACTION_OF_TRAIN, stratify=labels, random_state=RANDOM_SEED,
    )
    train_samples = [train_samples_all[i] for i in train_idx]
    val_samples = [train_samples_all[i] for i in val_idx]

    train_ds = FilteredImageDataset(train_samples, get_transforms(train=True))
    val_ds = FilteredImageDataset(val_samples, get_transforms(train=False))
    test_ds = FilteredImageDataset(test_samples, get_transforms(train=False))
    return train_ds, val_ds, test_ds, classes


train_ds, val_ds, test_ds, classes = load_datasets()
print("Classes used (alphabetical order = label indices):", classes)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


Classes used (alphabetical order = label indices): ['basal cell carcinoma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'squamous cell carcinoma']
Train: 1541  Val: 273  Test: 80


## 5. Model builders (8 transfer-learning architectures)

In [7]:
import torch.nn as nn
from torchvision import models


def build_model(name, num_classes):
    if name == "alexnet":
        m = models.alexnet(weights="IMAGENET1K_V1")
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "vgg16":
        m = models.vgg16(weights="IMAGENET1K_V1")
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "vgg19":
        m = models.vgg19(weights="IMAGENET1K_V1")
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "resnet18":
        m = models.resnet18(weights="IMAGENET1K_V1")
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "resnet101":
        m = models.resnet101(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "densenet121":
        m = models.densenet121(weights="IMAGENET1K_V1")
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m


def set_backbone_trainable(model, trainable: bool):
    for param in model.parameters():
        param.requires_grad = trainable
    last_layer = list(model.children())[-1]
    for param in last_layer.parameters():
        param.requires_grad = True


## 6. Training / evaluation loops

In [8]:
import copy
import time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()
            total_samples += images.size(0)
    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(probs.argmax(axis=1))
            all_probs.extend(probs)
    all_labels, all_preds, all_probs = np.array(all_labels), np.array(all_preds), np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="weighted")
    except ValueError:
        auc = float("nan")
    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(precision * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "AUC (%)": round(auc * 100, 2),
    }


def train_one_model(name, num_classes):
    print(f"\n=== Training {name} ===")
    model = build_model(name, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # Stage 1: feature extraction - freeze backbone, train head only
    set_backbone_trainable(model, trainable=False)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    for epoch in range(FEATURE_EXTRACT_EPOCHS):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        print(f"  [warmup {epoch+1}/{FEATURE_EXTRACT_EPOCHS}] train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

    # Stage 2: fine-tune the whole network
    set_backbone_trainable(model, trainable=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1 ** 0.5, patience=7, min_lr=0.5e-6
    )
    best_val_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    for epoch in range(FINE_TUNE_EPOCHS):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_loss)
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        print(f"  [finetune {epoch+1}/{FINE_TUNE_EPOCHS}] train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

    model.load_state_dict(best_state)
    metrics = evaluate(model, test_loader)
    metrics["Model"] = name
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"{name}_best.pt"))
    return metrics, model


## 7. Table 1 — Transfer Learning Models

Trains all 8 architectures in sequence. With 5 classes and reduced epochs
(2 warmup + 6 fine-tune), this should run noticeably faster than the full
9-class/20-epoch version — expect several minutes per model on a T4, not tens of minutes.

In [9]:
table1_results = []
for name in CNN_MODELS:
    start = time.time()
    metrics, _ = train_one_model(name, len(classes))
    metrics["Train Time (min)"] = round((time.time() - start) / 60, 1)
    table1_results.append(metrics)
    pd.DataFrame(table1_results).to_csv(os.path.join(OUTPUT_DIR, "table1_transfer_learning_results.csv"), index=False)

table1_df = pd.DataFrame(table1_results)
table1_df



=== Training alexnet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 171MB/s]


  [warmup 1/2] train_acc=0.523 val_acc=0.593
  [warmup 2/2] train_acc=0.611 val_acc=0.634
  [finetune 1/6] train_acc=0.635 val_acc=0.718
  [finetune 2/6] train_acc=0.718 val_acc=0.729
  [finetune 3/6] train_acc=0.745 val_acc=0.678
  [finetune 4/6] train_acc=0.749 val_acc=0.707
  [finetune 5/6] train_acc=0.788 val_acc=0.707
  [finetune 6/6] train_acc=0.796 val_acc=0.700

=== Training vgg16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 99.3MB/s]


  [warmup 1/2] train_acc=0.426 val_acc=0.553
  [warmup 2/2] train_acc=0.567 val_acc=0.612
  [finetune 1/6] train_acc=0.540 val_acc=0.670
  [finetune 2/6] train_acc=0.666 val_acc=0.608
  [finetune 3/6] train_acc=0.700 val_acc=0.645
  [finetune 4/6] train_acc=0.756 val_acc=0.634
  [finetune 5/6] train_acc=0.765 val_acc=0.689
  [finetune 6/6] train_acc=0.775 val_acc=0.740

=== Training vgg19 ===
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 114MB/s]


  [warmup 1/2] train_acc=0.240 val_acc=0.253
  [warmup 2/2] train_acc=0.310 val_acc=0.319
  [finetune 1/6] train_acc=0.602 val_acc=0.692
  [finetune 2/6] train_acc=0.751 val_acc=0.711
  [finetune 3/6] train_acc=0.811 val_acc=0.725
  [finetune 4/6] train_acc=0.848 val_acc=0.755
  [finetune 5/6] train_acc=0.881 val_acc=0.751
  [finetune 6/6] train_acc=0.888 val_acc=0.758

=== Training efficientnet_b0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 136MB/s] 


  [warmup 1/2] train_acc=0.271 val_acc=0.319
  [warmup 2/2] train_acc=0.367 val_acc=0.432
  [finetune 1/6] train_acc=0.563 val_acc=0.656
  [finetune 2/6] train_acc=0.707 val_acc=0.700
  [finetune 3/6] train_acc=0.773 val_acc=0.755
  [finetune 4/6] train_acc=0.795 val_acc=0.725
  [finetune 5/6] train_acc=0.829 val_acc=0.780
  [finetune 6/6] train_acc=0.846 val_acc=0.810


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Model,Train Time (min)
0,63.75,67.87,63.75,60.37,88.93,alexnet,4.0
1,66.25,67.15,66.25,64.57,89.71,vgg16,4.9
2,62.50,64.53,62.50,59.02,88.44,vgg19,5.2
3,61.25,62.88,61.25,54.71,91.89,resnet18,3.8
4,62.50,64.20,62.50,59.00,91.46,resnet50,4.2
5,65.00,68.72,65.00,61.14,93.69,resnet101,4.6
6,58.75,58.52,58.75,53.22,88.87,densenet121,4.3
7,62.50,68.13,62.50,56.47,92.44,efficientnet_b0,3.9


## 8. Table 2 — Classical Classifiers on Deep Features

Picks the best-performing CNN from Table 1, strips its classifier head, and feeds
the pooled feature vectors into 7 classical classifiers.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# Automatically pick the best model from Table 1; override manually if you prefer.
FEATURE_EXTRACTOR_MODEL = table1_df.sort_values("Accuracy (%)", ascending=False).iloc[0]["Model"]
print("Using feature extractor:", FEATURE_EXTRACTOR_MODEL)


def strip_classifier_head(model, name):
    if name == "alexnet" or name.startswith("vgg"):
        model.classifier[6] = nn.Identity()
    elif name.startswith("resnet"):
        model.fc = nn.Identity()
    elif name == "densenet121":
        model.classifier = nn.Identity()
    elif name == "efficientnet_b0":
        model.classifier[1] = nn.Identity()
    return model


def extract_features(loader, model):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            images = images.to(DEVICE)
            feats.append(model(images).cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


def evaluate_classifier(clf, X_test, y_test):
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds, average="weighted", zero_division=0)
    recall = recall_score(y_test, preds, average="weighted", zero_division=0)
    f1 = f1_score(y_test, preds, average="weighted", zero_division=0)
    try:
        probs = clf.predict_proba(X_test)
        auc = roc_auc_score(y_test, probs, multi_class="ovr", average="weighted")
    except (AttributeError, ValueError):
        auc = float("nan")
    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(precision * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "AUC (%)": round(auc * 100, 2),
    }


feat_model = build_model(FEATURE_EXTRACTOR_MODEL, len(classes))
feat_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, f"{FEATURE_EXTRACTOR_MODEL}_best.pt"), map_location=DEVICE))
feat_model = strip_classifier_head(feat_model, FEATURE_EXTRACTOR_MODEL).to(DEVICE)

train_loader_noshuffle = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("Extracting deep features for train set...")
X_train, y_train = extract_features(train_loader_noshuffle, feat_model)
print("Extracting deep features for test set...")
X_test, y_test = extract_features(test_loader, feat_model)
print("Feature vector size:", X_train.shape[1])

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": CalibratedClassifierCV(LinearSVC(max_iter=5000)),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED),
    "XGBoost": XGBClassifier(n_estimators=300, eval_metric="mlogloss", random_state=RANDOM_SEED),
}

table2_results = []
for name, clf in classifiers.items():
    print(f"\n=== Training {name} on deep features ===")
    clf.fit(X_train, y_train)
    metrics = evaluate_classifier(clf, X_test, y_test)
    metrics["Feature Extractor"] = "Deep Features"
    metrics["Classifier"] = name
    table2_results.append(metrics)
    pd.DataFrame(table2_results).to_csv(os.path.join(OUTPUT_DIR, "table2_classifier_results.csv"), index=False)

table2_df = pd.DataFrame(table2_results)
table2_df


Using feature extractor: vgg16
Extracting deep features for train set...
Extracting deep features for test set...
Feature vector size: 4096

=== Training Logistic Regression on deep features ===

=== Training Decision Tree on deep features ===

=== Training Random Forest on deep features ===

=== Training K-Nearest Neighbors (KNN) on deep features ===

=== Training Linear SVM on deep features ===


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



=== Training RBF-SVM on deep features ===

=== Training XGBoost on deep features ===


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Feature Extractor,Classifier
0,58.75,63.10,58.75,55.38,88.57,Deep Features,Logistic Regression
1,48.75,46.55,48.75,44.91,67.97,Deep Features,Decision Tree
2,61.25,64.34,61.25,56.70,89.07,Deep Features,Random Forest
3,61.25,66.20,61.25,55.51,82.91,Deep Features,K-Nearest Neighbors (KNN)
4,61.25,69.21,61.25,58.54,87.91,Deep Features,Linear SVM
5,60.00,61.75,60.00,55.67,89.90,Deep Features,RBF-SVM
6,61.25,63.73,61.25,56.84,86.27,Deep Features,XGBoost


## 9. Table 3 — Computational Efficiency Comparison

Parameters, model size, FLOPs, and inference time for all 8 architectures (accuracy pulled from Table 1).

In [11]:
from thop import profile

NUM_TIMING_RUNS = 50


def model_size_mb(checkpoint_path):
    return os.path.getsize(checkpoint_path) / (1024 ** 2)


def measure_inference_time_ms(model, input_size):
    model.eval()
    dummy = torch.randn(1, 3, input_size, input_size).to(DEVICE)
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(NUM_TIMING_RUNS):
            model(dummy)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return ((time.time() - start) / NUM_TIMING_RUNS) * 1000


table3_results = []
for name in CNN_MODELS:
    print(f"\n=== Benchmarking {name} ===")
    model = build_model(name, len(classes)).to(DEVICE)
    checkpoint_path = os.path.join(OUTPUT_DIR, f"{name}_best.pt")
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    size_mb = round(model_size_mb(checkpoint_path), 2)

    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    inference_ms = measure_inference_time_ms(model, IMAGE_SIZE)

    accuracy_row = table1_df[table1_df["Model"] == name]
    accuracy = accuracy_row.iloc[0]["Accuracy (%)"] if not accuracy_row.empty else None

    table3_results.append({
        "Model": name,
        "Parameters (M)": round(params / 1e6, 2),
        "Model Size (MB)": size_mb,
        "FLOPs (G)": round(flops / 1e9, 2),
        "Inference Time (ms)": round(inference_ms, 2),
        "Accuracy (%)": accuracy,
    })
    pd.DataFrame(table3_results).to_csv(os.path.join(OUTPUT_DIR, "table3_efficiency_results.csv"), index=False)

table3_df = pd.DataFrame(table3_results)
table3_df



=== Benchmarking alexnet ===

=== Benchmarking vgg16 ===

=== Benchmarking vgg19 ===

=== Benchmarking resnet18 ===

=== Benchmarking resnet50 ===

=== Benchmarking resnet101 ===

=== Benchmarking densenet121 ===

=== Benchmarking efficientnet_b0 ===


,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,alexnet,57.02,217.54,0.71,2.09,63.75
1,vgg16,134.28,512.25,15.47,10.44,66.25
2,vgg19,139.59,532.51,19.63,12.46,62.50
3,resnet18,11.18,42.72,1.82,2.27,61.25
4,resnet50,23.52,90.02,4.13,5.77,62.50
5,resnet101,42.51,162.77,7.86,11.74,65.00
6,densenet121,6.96,27.13,2.90,15.35,58.75
7,efficientnet_b0,4.01,15.60,0.41,7.97,62.50


## 10. Download the results

CSV files matching your three Word-doc tables are in `./results/`.

In [12]:
from google.colab import files

for fname in ["table1_transfer_learning_results.csv", "table2_classifier_results.csv", "table3_efficiency_results.csv"]:
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        files.download(path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>